# ML-00 ML Input Export from 03 Feature Output

- `data/preprocessing/03_ml_feature_process.ipynb`가 만든 `ml_exp00.parquet` 단일 산출물을 사용, `ml_00_ml_*.py` 모듈에 넣을 수 있는 입력 파일 세트를 생성 한다. 
- 이 노트북은 **학습/검증/test를 실행하지 않는다.** 

##  생성 목표
1. `ml_exp00.parquet` 단일 파일의 schema와 `split` 컬럼 확인
2. `ml_feature_columns.csv`의 `used_in_ml="TRUE"` feature 목록 확인
3. 단일 parquet를 `split` 컬럼 기준으로 train/val/test 3개 parquet 파일로 분리
4. `ml_feature_columns.csv`를 `fb_outputs` run directory에 복사
5. 후속 `ml_00_ml_*.py` 모듈에 넘길 경로를 manifest로 저장

## 실행 범위

포함:
- `ml_00_fb_utils.py` 기준 프로젝트 경로 설정
- `ml_00_fb_build.split_single_parquet_by_existing_split()` 실행
- `fb_outputs` 후보 산출물 존재 여부와 schema 확인
- `fb_outputs/manifest.json` 저장

제외:
- XGBoost 학습
- validation threshold 선택
- final test 평가
- metric 생성

## 실행 전 주의

- 원본 `data/processed/ml_features/ml_exp00.parquet`는 수정하지 않는다.
- 원본 `ml_feature_columns.csv`는 수정하지 않고 output directory로 복사한다.
- split parquet는 `ml/ml-00_baseline/fb_outputs/` 아래 새 run directory에 저장한다. 
- 사람이 `fb_outputs` 산출물을 직접 검토한 뒤 승인된 파일만 `ml_inputs`로 복사한다.
- 기본 overwrite는 `False`입니다. 같은 설정으로 재실행하려면 `RUN_ID`를 바꾸는 것을 권장한다. 
- 이 노트북이 만든 parquet/CSV/JSON 산출물은 commit 대상이 아니다

# 00_경로 및 환경설정

In [ ]:
from __future__ import annotations
import json
import shutil
import sys
from pathlib import Path
import pandas as pd

# ============================================================
# 0.1 Runtime Settings
# ============================================================
# 실행 환경: ML 담당자 기준
# - Kernel          : Local WSL
# - Code repo       : local Git repo
# - Input features  : Google Drive Graph_AML
# - Output artifacts: local Git repo ml/ml-00_baseline/fb_outputs/
SEED = 42 #랜덤 시드 고정

# ------------------------------------------------------------
# Root Paths
# ------------------------------------------------------------
# 다른 환경에서 실행할 경우 보통 이 두 경로만 수정한다.
LOCAL_REPO_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()
DRIVE_REPO_ROOT = Path("/mnt/g/내 드라이브/Graph_AML").resolve()

# ------------------------------------------------------------
# Input Paths
# ------------------------------------------------------------
# 입력 feature parquet/csv는 Google Drive의 processed/ml_features에서 읽는다.
ML_00_IN_DRIVE_PROCESSED_DIR = DRIVE_REPO_ROOT / "data" / "processed"
ML_00_IN_SOURCE_DATA_DIR = ML_00_IN_DRIVE_PROCESSED_DIR / "ml_features"

# ------------------------------------------------------------
# Output Paths
# ------------------------------------------------------------
# ML-00은 전처리 폴더에서 직접 입력을 가져오므로 fb_inputs를 쓰지 않는다.
# feature_build 모듈이 생성한 후보 산출물은 fb_outputs에 저장한다.
ML_00_BASE_DIR = LOCAL_REPO_ROOT / "ml" / "ml-00_baseline"
ML_00_FB_OUTPUTS_DIR = ML_00_BASE_DIR / "fb_outputs"

# 사람이 검토 후 승인한 ML 입력 파일은 fb_outputs에서 ml_inputs로 복사한다.
ML_00_ML_INPUTS_DIR = ML_00_BASE_DIR / "ml_inputs"

# ------------------------------------------------------------
# Module Code Paths
# ------------------------------------------------------------
# local Git repo에서 코드 모듈을 import한다.
ML_00_MODULE_FB_DIR = LOCAL_REPO_ROOT / "ml" / "ml-00_baseline" / "feature_build"
ML_00_MODULE_TVT_DIR = LOCAL_REPO_ROOT / "ml" / "ml-00_baseline" / "train_val_test"


# ============================================================
# 0.2 Path Validation
# ============================================================
def require_dir(path: Path, name: str) -> None:
    """필수 디렉터리가 없으면 시작 단계에서 중단한다."""
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")
    
# 시작 단계에서 반드시 존재해야 하는 디렉터리 목록이다.
# 새로 생성될 하위 산출물 디렉터리(FB_OUTPUT_DIR)는 여기서 검증하지 않는다.
REQUIRED_DIRS = {
    "LOCAL_REPO_ROOT": LOCAL_REPO_ROOT,
    "DRIVE_REPO_ROOT": DRIVE_REPO_ROOT,
    "ML_00_IN_SOURCE_DATA_DIR": ML_00_IN_SOURCE_DATA_DIR,
    "ML_00_BASE_DIR": ML_00_BASE_DIR,
    "ML_00_FB_OUTPUTS_DIR": ML_00_FB_OUTPUTS_DIR,
    "ML_00_ML_INPUTS_DIR": ML_00_ML_INPUTS_DIR,
    "ML_00_MODULE_FB_DIR": ML_00_MODULE_FB_DIR,
    "ML_00_MODULE_TVT_DIR": ML_00_MODULE_TVT_DIR,
}
for name, path in REQUIRED_DIRS.items():
    require_dir(path, name)
    
# ============================================================
# 0.3 Import Path Setup
# ============================================================
# local train_val_test/ml_00_ml_io.py와 feature_build/ml_00_fb_* 모듈을 우선 import한다.
IMPORT_DIRS = [str(ML_00_MODULE_TVT_DIR), str(ML_00_MODULE_FB_DIR)]
IMPORT_MODULES = ("ml_00_fb_utils", "ml_00_fb_build", "ml_00_fb_io", "ml_00_ml_io")

# 중복 경로를 제거한 뒤 맨 앞에 삽입해 import 우선순서를 고정한다.
sys.path = [path for path in sys.path if path not in IMPORT_DIRS]
sys.path[:0] = IMPORT_DIRS

# 노트북 재실행 시 수정된 local 모듈을 다시 읽도록 import cache를 제거한다.
for module_name in IMPORT_MODULES:
    sys.modules.pop(module_name, None)
import ml_00_fb_utils
from ml_00_fb_build import split_single_parquet_by_existing_split
from ml_00_fb_io import parquet_columns
from ml_00_ml_io import (
    check_feature_columns_file,
    feature_columns_hash,
    normalize_feature_columns_file,
)
ml_00_fb_utils.set_seed(SEED, use_torch=False)

# ============================================================
# 0.4 Configuration Summary
# ============================================================
print("=" * 60)
print(f"SEED                           : {SEED}")
print("Kernel                         : Local WSL")
print("INPUT_DATA_SOURCE              : hardcoded drive")
print("-" * 60)
print(f"LOCAL_REPO_ROOT                : {LOCAL_REPO_ROOT}")
print(f"DRIVE_REPO_ROOT                : {DRIVE_REPO_ROOT}")
print("-" * 60)
print(f"ML_00_IN_DRIVE_PROCESSED_DIR   : {ML_00_IN_DRIVE_PROCESSED_DIR}")
print(f"ML_00_IN_SOURCE_DATA_DIR       : {ML_00_IN_SOURCE_DATA_DIR}")
print("-" * 60)
print(f"ML_00_BASE_DIR                 : {ML_00_BASE_DIR}")
print(f"ML_00_FB_OUTPUTS_DIR           : {ML_00_FB_OUTPUTS_DIR}")
print("-" * 60)
print(f"ML_00_ML_INPUTS_DIR            : {ML_00_ML_INPUTS_DIR}")
print("-" * 60)
print(f"ML_00_MODULE_FB_DIR            : {ML_00_MODULE_FB_DIR}")
print(f"ML_00_MODULE_TVT_DIR           : {ML_00_MODULE_TVT_DIR}")
print("=" * 60 + "\n")

# 01_실행 설정
- 입력 산출물 위치, output directory, overwrite 정책을 제어

In [ ]:
# ============================================================
# 1.1 Run identifiers 
# ============================================================
# EXPORT_EXPERIMENT_ID / RUN_ID: 출력 폴더명을 구성하는 식별자
EXPORT_EXPERIMENT_ID = "ml_00_test"
RUN_ID = "run_00"

# ARTIFACT_PREFIX:
# - split parquet와 split summary 파일명에만 붙는 prefix
# - 예: ml_00_test__run_00_Xy_train.parquet
ARTIFACT_PREFIX = f"{EXPORT_EXPERIMENT_ID}__{RUN_ID}"

# 정답 라벨 컬럼명
LABEL_COL = "label"

# 메타데이터로 반드시 존재해야 하는 컬럼명. 이 컬럼들은 feature column과 별도로 존재해야 한다.
REQUIRED_EXPORT_META_COLUMNS = {"tx_id", "timestamp", "split"}

# ============================================================
# 1.2 Source input  file and paths -> 중요 입력 자료는 파일명 별도 지정
# ============================================================
# SOURCE_PARQUET_FILE / FEATURE_COLUMNS_SOURCE_FILE:
# ML_00_IN_SOURCE_DATA_DIR 아래에서 읽을 parquet 파일명과 feature column CSV 파일명
SOURCE_PARQUET_FILE = "ml_exp00.parquet" 
FEATURE_COLUMNS_SOURCE_FILE = "ml_feature_columns.csv"

# ML_00_IN_SOURCE_DATA_DIR는 bootstrap 셀에서 결정된다.
# 필요한 입력 파일: {SOURCE_PARQUET_FILE} + {FEATURE_COLUMNS_SOURCE_FILE}
INPUT_SINGLE_PARQUET_PATH = ML_00_IN_SOURCE_DATA_DIR / SOURCE_PARQUET_FILE
FEATURE_COLUMNS_SOURCE_DIR = ML_00_IN_SOURCE_DATA_DIR
FEATURE_COLUMNS_SOURCE_FILE_PATH = FEATURE_COLUMNS_SOURCE_DIR / FEATURE_COLUMNS_SOURCE_FILE

# ============================================================
# 1.3 Export output paths -> 출력산출물은 경로랑 파일명 묶어서 지정
# ============================================================
# FB_OUTPUT_DIR: feature_build 모듈이 생성한 후보 산출물 저장 위치
# ML_INPUT_DIR: 사람이 검토 후 fb_outputs에서 복사할 승인 ML 입력 위치
FB_OUTPUT_DIR = ML_00_FB_OUTPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID
ML_INPUT_DIR = ML_00_ML_INPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID

# fb_outputs에 생성될 split export 후보 산출물 경로
FB_OUTPUT_TRAIN_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_train.parquet"
FB_OUTPUT_VAL_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_val.parquet"
FB_OUTPUT_TEST_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_test.parquet"

# split-only 실행 요약 파일 경로
FB_OUTPUT_SPLIT_SUMMARY_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_split_summary.csv"
FB_OUTPUT_SPLIT_BUILD_SUMMARY_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_split_existing_summary.json"

# fb_outputs에 복사/정규화할 feature column CSV 후보 산출물 경로
# 사용자가 검토할 파일이므로 파일명은 짧게 유지한다.
FB_OUTPUT_FEATURE_COLUMNS_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_ml_feature_columns.csv"

# 이 노트북 실행에서 사용한 입력/출력/설정 정보를 기록하는 manifest 경로
FB_OUTPUT_MANIFEST_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_manifest.json"

# 사람이 검토 후 ml_inputs로 복사/저장할 승인 ML 입력 대상 경로
# train/val/test parquet는 후보 파일명 그대로 복사하고, feature column CSV는 검토 후 _approve 파일명으로 저장한다.
ML_INPUT_TRAIN_PATH = ML_INPUT_DIR / FB_OUTPUT_TRAIN_PATH.name
ML_INPUT_VAL_PATH = ML_INPUT_DIR / FB_OUTPUT_VAL_PATH.name
ML_INPUT_TEST_PATH = ML_INPUT_DIR / FB_OUTPUT_TEST_PATH.name
ML_INPUT_FEATURE_COLUMNS_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_ml_feature_columns_approve.csv"

# ============================================================
# 1.4 Execution switches
# ============================================================
# True인 항목만 실행한다. 경로 preview 또는 설정 점검만 할 때는 필요한 항목을 False로 바꾼다.
RUN_SPLIT_EXPORT = True
COPY_FEATURE_COLUMNS = True
SAVE_MANIFEST = True

# test label 분포 표시 여부: final test 전 기본 숨김
# - False: test의 label count/positive_rate를 화면에서 숨김
# - True: 사용자가 명시 승인한 경우에만 test label 분포 표시
SHOW_TEST_LABEL_DISTRIBUTION = False

# ============================================================
# 1.5 Overwrite policy
# ============================================================
# False: 기존 산출물이 있으면 실행을 중단한다. 재실행 시 RUN_ID 변경을 권장한다.
# True: 기존 산출물 overwrite를 허용한다. 실험 재현성 추적이 어려워질 수 있으므로 필요한 경우에만 사용한다.
OVERWRITE_POLICY = {
    "split_export": False,
    "feature_columns_copy": False,
    "manifest": False,
}
# ============================================================
# 1.6 Configuration preview
# ============================================================
print("=" * 80)
print("ML_00_IN_SOURCE_DATA_DIR         :", ML_00_IN_SOURCE_DATA_DIR)
print("INPUT_SINGLE_PARQUET_PATH        :", INPUT_SINGLE_PARQUET_PATH)
print("SOURCE_PARQUET_FILE              :", SOURCE_PARQUET_FILE)
print("-" * 80)
print("FEATURE_COLUMNS_SOURCE_DIR       :", FEATURE_COLUMNS_SOURCE_DIR)
print("FEATURE_COLUMNS_SOURCE_FILE_PATH :", FEATURE_COLUMNS_SOURCE_FILE_PATH)
print("FEATURE_COLUMNS_SOURCE_FILE      :", FEATURE_COLUMNS_SOURCE_FILE)
print("-" * 80)
print("EXPORT_EXPERIMENT_ID          :", EXPORT_EXPERIMENT_ID)
print("RUN_ID                        :", RUN_ID)
print("ARTIFACT_PREFIX               :", ARTIFACT_PREFIX)
print("LABEL_COL                     :", LABEL_COL)
print("REQUIRED_EXPORT_META_COLUMNS  :", REQUIRED_EXPORT_META_COLUMNS)
print("-" * 80)
print("ML_00_FB_OUTPUTS_DIR          :", ML_00_FB_OUTPUTS_DIR)
print("FB_OUTPUT_DIR                 :", FB_OUTPUT_DIR)
print("FB_OUTPUT_TRAIN_PATH          :", FB_OUTPUT_TRAIN_PATH)
print("FB_OUTPUT_VAL_PATH            :", FB_OUTPUT_VAL_PATH)
print("FB_OUTPUT_TEST_PATH           :", FB_OUTPUT_TEST_PATH)
print("FB_OUTPUT_SPLIT_SUMMARY_PATH  :", FB_OUTPUT_SPLIT_SUMMARY_PATH)
print("FB_OUTPUT_SPLIT_BUILD_SUMMARY_PATH :", FB_OUTPUT_SPLIT_BUILD_SUMMARY_PATH)
print("FB_OUTPUT_FEATURE_COLUMNS_PATH :", FB_OUTPUT_FEATURE_COLUMNS_PATH)
print("FB_OUTPUT_MANIFEST_PATH       :", FB_OUTPUT_MANIFEST_PATH)
print("-" * 80)
print("ML_00_ML_INPUTS_DIR           :", ML_00_ML_INPUTS_DIR)
print("ML_INPUT_DIR                  :", ML_INPUT_DIR)
print("ML_INPUT_TRAIN_PATH           :", ML_INPUT_TRAIN_PATH)
print("ML_INPUT_VAL_PATH             :", ML_INPUT_VAL_PATH)
print("ML_INPUT_TEST_PATH            :", ML_INPUT_TEST_PATH)
print("ML_INPUT_FEATURE_COLUMNS_PATH :", ML_INPUT_FEATURE_COLUMNS_PATH)
print("=" * 80)

# 02_입력 계약 검증 함수 정의
- `ml_exp00.parquet` 과 `ml_feature_columns.csv`가 후속 ML 입력 계약을 만족하는지 확인하는 함수를 정의한다.  
- 실제 검증은 export 사본을 만든 뒤 `FB_OUTPUT_FEATURE_COLUMNS_PATH` 기준으로 수행한다.

In [ ]:
# ============================================================
# 2.1 require_file
# ============================================================
def require_file(path: Path, label: str) -> Path:
    """필수 파일 존재 여부를 fail-fast로 확인하고 resolve된 Path를 반환한다."""
    path = Path(path).resolve()      # 입력값이 str이어도 Path로 통일하고, 실행 위치 기준 절대 경로로 변환한다.
    if not path.exists():            # 경로 자체가 없으면 후속 로직을 실행하지 않고 즉시 중단한다.
        raise FileNotFoundError(
            f"{label} not found: {path}\n"
            "입력 산출물이 생성/복사되었는지, 또는 설정 셀의 경로가 맞는지 확인하세요."
        )
    if not path.is_file():           # 경로는 존재하지만 파일이 아니라 디렉터리 등인 경우도 입력 오류로 처리한다.
        raise FileNotFoundError(f"{label} is not a file: {path}")
    return path                      # 검증이 끝난 절대 경로를 반환해 이후 단계에서 같은 기준 경로를 사용하도록 한다.

# ============================================================
# 2.2 guard_output_overwrite
# ============================================================
def guard_output_overwrite(path: Path, *, overwrite: bool, label: str) -> Path:
    """
    노트북이 직접 생성하는 보조 산출물의 overwrite를 차단한다.
    사용 위치:
    - feature columns CSV 사본 저장 전
    - manifest JSON 저장 전
    제외 대상:
    - split parquet 3개
    - split summary 파일
    위 파일들은 ml_00_fb_build.split_single_parquet_by_existing_split() 내부 overwrite 정책으로 관리한다.
    """
    path = Path(path).resolve()      # 출력 경로도 Path로 통일하고 절대 경로로 변환해 에러 메시지와 저장 위치를 명확히 한다.
    
    # 기존 파일이 있는데 overwrite=False이면 실험 산출물 덮어쓰기를 막고 즉시 중단한다.
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"Existing {label} found: {path}. "
            "Set overwrite=True or change RUN_ID to create a new output directory."
        )
    return path                      # 파일이 없거나 overwrite=True이면 호출부에서 저장을 계속 진행할 수 있도록 경로를 반환한다.

# ============================================================
# 2.3 preview_input_contract
# ============================================================
def preview_input_contract(
    *,
    input_parquet_path: Path,
    feature_columns_path: Path,
    label_col: str,
    verbose: bool = True,
) -> list[str]:
    """
    입력 parquet과 feature column CSV가 ML input export 계약을 만족하는지 확인한다.
    확인 항목:
    - parquet / CSV 파일 존재 여부
    - parquet 필수 meta 컬럼 존재 여부
    - ml_feature_columns.csv strict 검증
    - used_in_ml="TRUE" feature가 parquet schema에 모두 존재하는지
    """
    # 학습 입력 parquet, feature column CSV 경로가 실제 파일인지 먼저 확인하고 절대 경로로 통일한다.
    input_parquet_path = require_file(input_parquet_path, "input_parquet_path")
    feature_columns_path = require_file(feature_columns_path, "feature_columns_path")
    parquet_column_names = parquet_columns(input_parquet_path)  # parquet schema에서 컬럼 이름만 읽어온다.
    parquet_column_set = set(parquet_column_names)              # parquet 컬럼 목록을 set으로 변환한다.
    
    # ---------------------------------------------------------------------
    required_meta_columns = REQUIRED_EXPORT_META_COLUMNS | {label_col}        # ML 입력 parquet에 meta 컬럼과 label 컬럼을 합친다.
    missing_meta_columns = sorted(required_meta_columns - parquet_column_set) # 필수 meta/label 컬럼 중 parquet에 없는 컬럼을 목록으로 만든다.
    # 필수 컬럼이 하나라도 없으면 중단한다.
    if missing_meta_columns:
        raise ValueError(
            f"input parquet is missing required meta columns: {missing_meta_columns}"
        )
        
    # ---------------------------------------------------------------------
    # feature column CSV가 계약을 만족하는지 strict 모드로 검증한다.
    # 여기서 used_in_ml 값, label 누수 위험, 필수 컬럼 등을 확인한다.
    feature_check = check_feature_columns_file(
        feature_columns_path,
        label_col=label_col,
        strict=True,
    )
    feature_columns = feature_check.selected_columns                              # used_in_ml="TRUE" feature 목록을 가져온다.
    missing_features = [
        column for column in feature_columns if column not in parquet_column_set  # 선택된 feature 중 실제 parquet schema에 없는 컬럼을 찾는다.
    ]
    # feature CSV에는 있는데 parquet에는 없는 컬럼이 있으면 중단한다.
    if missing_features:
        raise ValueError(
            'used_in_ml="TRUE" feature columns are missing from input parquet. '
            f"missing={missing_features[:30]}, missing_count={len(missing_features)}"
        )
        
    # ---------------------------------------------------------------------
    # verbose=True이면 검증 결과와 핵심 fingerprint를 출력한다.
    if verbose:
        summary = {
            "input_parquet_path": input_parquet_path,
            "feature_columns_path": feature_columns_path,
            "input parquet column_count": len(parquet_column_names),
            "selected feature_count": feature_check.selected_count,
            "feature_columns_hash": feature_columns_hash(feature_columns),
            "first_features": feature_columns[:10],
            "input contract": "OK",
        }
        for key, value in summary.items():
            print(f"{key:<28}: {value}")
    return feature_columns          # 검증을 통과한 최종 feature 목록을 후속 split/export/train 단계에서 사용한다.

print("input contract helper functions loaded")

# 03_`ml_exp00.parquet` 파일의 train/val/test 라벨 분포 확인
- `split`, `label` 두 컬럼만 읽어서 원본 단일 parquet가 train/val/test를 모두 포함하는지 확인한다.

In [ ]:
def summarize_split_labels(
    parquet_path: Path,
    *,
    expected_split: str | None = None,
    label_col: str = LABEL_COL,
) -> pd.DataFrame:
    """
    parquet의 split별 row 수와 label 분포를 확인한다.
    경로 정책
    --------
    - parquet_path는 앞 셀에서 정의한 경로 변수를 전달한다.
    - source 단일 parquet 확인 시: INPUT_SINGLE_PARQUET_PATH 예: {ML_00_IN_SOURCE_DATA_DIR}/{SOURCE_PARQUET_FILE}
    - export 후 split parquet 확인 시: FB_OUTPUT_TRAIN_PATH, FB_OUTPUT_VAL_PATH, FB_OUTPUT_TEST_PATH  예: {FB_OUTPUT_DIR}/{ARTIFACT_PREFIX}_Xy_train.parquet
    사용 목적
    --------
    - split export 전 원본 단일 parquet가 train/val/test를 모두 포함하는지 확인한다.
    - split export 후 개별 train/val/test parquet가 자기 split만 포함하는지 확인한다.
    - 후속 ML 학습 전에 각 split의 positive/negative 분포와 양 클래스 존재 여부를 확인한다.
    확인 항목
    --------
    - parquet 파일 존재 여부
    - split 컬럼 결측/공백 여부
    - split 값이 train/val/test 정책을 만족하는지
    - label_col이 0/1 이진값인지
    - split별 row 수, positive count, negative count, positive rate
    - 각 split에 양/음 클래스가 모두 존재하는지
    주의
    ----
    - split은 train/val/test만 허용한다.
    - label은 0/1 이진값만 허용한다.
    - 전체 parquet를 읽지 않고 split, label_col 두 컬럼만 읽는다.
    """
    # 1. parquet 파일이 실제로 존재하는지 확인하고, 이후 출력/에러 메시지에 쓸 절대 경로로 통일한다.
    parquet_path = require_file(parquet_path, 
                                "split label summary parquet")
    
    # 2. 전체 parquet를 읽지 않고 검증에 필요한 split과 label 컬럼만 읽는다.
    df = pd.read_parquet(parquet_path, 
                         columns=["split", label_col])
    
    # 3. split 결측이 있으면 중단한다.
    if df["split"].isna().any():
        raise ValueError(f"split contains missing values. path={parquet_path}")
    
    # 4. split 표기는 대소문자와 앞뒤 공백만 정규화한다. 예: " Train " -> "train"
    df["split"] = df["split"].astype("string").str.strip().str.lower()
    
    # 5. 공백만 있던 split 값은 정규화 후 빈 문자열이 되므로 유효하지 않은 split으로 처리한다.
    if (df["split"] == "").any():
        raise ValueError(f"split contains blank values. path={parquet_path}")
    
    # 6. 이 파이프라인에서 허용하는 split 이름은 train/val/test로 고정한다.
    allowed_splits = {"train", "val", "test"}
    
    # 7. expected_split=None이면 source 단일 parquet 검증 모드이고, 
    # 값이 있으면 개별 split parquet 검증 모드다.
    expected = None if expected_split is None else expected_split.strip().lower()
    
    # 8. expected_split 인자가 들어온 경우 train/val/test 중 하나인지 먼저 확인한다.
    if expected is not None and expected not in allowed_splits:
        raise ValueError(
            f"expected_split must be one of {sorted(allowed_splits)}. "
            f"expected_split={expected_split!r}"
        )
        
    # 9. source 단일 parquet는 train/val/test 전체가 있어야 하고, 
    # 개별 split parquet는 expected split 하나만 있어야 한다.
    expected_values = {expected} if expected is not None else allowed_splits
    
    # 10. parquet 내부에서 실제 발견된 split 값 목록을 set으로 만든다.
    observed_splits = set(df["split"].unique().tolist())
    
    # 11. 실제 split 구성이 기대값과 다르면 split export 또는 입력 파일 선택이 잘못된 것이다.
    # 예: train parquet에 val/test row가 섞였거나, source parquet에 test split이 없는 경우.
    if observed_splits != expected_values:
        raise ValueError(
            f"unexpected split values in {parquet_path}: "
            f"expected={sorted(expected_values)}, observed={sorted(observed_splits)}"
        )
        
    # 12. label 컬럼을 숫자로 변환한다. 숫자로 해석할 수 없는 값이 있으면 여기서 실패한다.
    labels = pd.to_numeric(df[label_col], errors="raise")
    
    # 13. 결측 label을 제외하고 실제 등장한 label 값을 확인한다.
    observed_labels = set(labels.dropna().unique().tolist())
    
    # 14. label은 결측 없이 0/1 이진값만 허용한다.
    if labels.isna().any() or not observed_labels <= {0, 1}:
        raise ValueError(
            f"label must be binary 0/1. "
            f"path={parquet_path}, observed={sorted(observed_labels)}"
        )
        
    # 15. 출력 row 순서를 고정한다.
    # expected_split이 있으면 해당 split만, 없으면 train/val/test 순서로 출력한다.
    split_order = [expected] if expected is not None else ["train", "val", "test"]
    
    # 16. split별 요약 row를 누적할 리스트다.
    rows = []
    
    # 17. 정해진 split 순서대로 row 수와 label 분포를 계산한다.
    for split_name in split_order:
        part = df[df["split"] == split_name]   # 18. 현재 split에 해당하는 row만 선택한다.
        part_labels = labels.loc[part.index]   # 19. labels는 원본 df와 같은 index를 유지하므로 part.index로 현재 split의 label만 가져온다.
        positive_count = int((part_labels == 1).sum()) # 20. 현재 split의 positive label 개수를 계산한다.
        negative_count = int((part_labels == 0).sum()) # 21. 현재 split의 negative label 개수를 계산한다.
        row_count = int(len(part))                     # 22. 현재 split의 전체 row 수를 계산한다.
        # 23. split별 row 수, label 분포, 양성 비율, 양/음 클래스 존재 여부를 기록한다.
        rows.append(
            {
                "split": split_name,
                "rows": row_count,
                "positive_count": positive_count,
                "negative_count": negative_count,
                "positive_rate": float(positive_count / row_count),
                "both_classes": 0 < positive_count < row_count,
            }
        )
    return pd.DataFrame(rows)     # 24. 리스트 형태의 요약 결과를 DataFrame으로 변환해 반환한다.


def hide_test_label_distribution(summary: pd.DataFrame) -> pd.DataFrame:
    """SHOW_TEST_LABEL_DISTRIBUTION=False이면 test label 분포만 화면에서 숨긴다."""
    display_summary = summary.copy()
    if SHOW_TEST_LABEL_DISTRIBUTION or "split" not in display_summary.columns:
        return display_summary
    hidden_columns = [
        column
        for column in [
            "positive_count",
            "negative_count",
            "positive_rate",
            "both_classes",
            "label_0_count",
            "label_1_count",
        ]
        if column in display_summary.columns
    ]
    if hidden_columns:
        test_mask = display_summary["split"].astype("string").str.strip().str.lower() == "test"
        display_summary[hidden_columns] = display_summary[hidden_columns].astype("object")
        display_summary.loc[test_mask, hidden_columns] = "[hidden]"
    return display_summary


ML_00_INPUT_SPLIT_LABEL_SUMMARY = summarize_split_labels(
    INPUT_SINGLE_PARQUET_PATH,
    label_col=LABEL_COL,
)

display(hide_test_label_distribution(ML_00_INPUT_SPLIT_LABEL_SUMMARY))

# 04_`ml_exp00.parquet` -> ML 입력 parquet 생성

- `ml_00_fb_build.split_single_parquet_by_existing_split()`만 사용한다.
- 이 함수는 feature를 새로 계산하지 않고 기존 `split` 컬럼을 기준으로 파일을 나눈다.

In [ ]:
def export_split_parquets():
    """
    source 단일 parquet를 후속 ML 모듈 입력용 split parquet 3개로 분리한다.
    경로 정책
    --------
    - 입력: INPUT_SINGLE_PARQUET_PATH 예: {ML_00_IN_SOURCE_DATA_DIR}/{SOURCE_PARQUET_FILE}
    - 출력: FB_OUTPUT_DIR 예: {ML_00_FB_OUTPUTS_DIR}/{EXPORT_EXPERIMENT_ID}/{RUN_ID}
    처리 내용
    --------
    - 입력 parquet에 이미 존재하는 split 컬럼을 기준으로 train/val/test 파일만 export한다.
    - 출력 파일명 prefix는 ARTIFACT_PREFIX를 사용한다.
    overwrite 정책
    ---------------
    - OVERWRITE_POLICY["split_export"]가 False이면 기존 split parquet가 있을 때 중단한다.
    - 같은 설정으로 재실행할 때는 RUN_ID 변경을 권장한다.
    """
    # 1. split export 스위치가 꺼져 있으면 파일 생성 없이 후속 검증 단계로 넘긴다.
    if not RUN_SPLIT_EXPORT:
        print("Split export skipped. Existing split files will be validated if present.")
        return None
    
    # 2. 입력 단일 parquet가 실제 파일인지 먼저 확인하고, 절대 경로 Path로 통일한다.
    input_single_parquet = require_file(INPUT_SINGLE_PARQUET_PATH, "INPUT_SINGLE_PARQUET_PATH")
    
    # 3. split export overwrite 정책만 꺼내서 호출부에서 어떤 정책을 쓰는지 명확히 한다.
    split_overwrite = OVERWRITE_POLICY["split_export"]
    
    # 4. FB_OUTPUT_DIR는 여기서 직접 mkdir하지 않는다.
    # split_single_parquet_by_existing_split() 내부에서 기존 산출물 overwrite 여부를 먼저 확인한 뒤 output_dir를 생성한다.
    result = split_single_parquet_by_existing_split(
        input_path=input_single_parquet,      # 검증된 source 단일 parquet 경로를 넘긴다.
        output_dir=FB_OUTPUT_DIR,             # train/val/test 후보 산출물이 저장될 fb_outputs run directory다.
        base_dir=LOCAL_REPO_ROOT,             # 상대 경로가 들어올 경우 local repo root 기준으로 해석하게 한다.
        experiment_id=ARTIFACT_PREFIX,        # 출력 파일명 prefix로 사용된다.
        overwrite=split_overwrite,            # 기존 split 산출물 덮어쓰기 허용 여부다.
        tx_id_col="tx_id",                    # 거래 ID 컬럼명이다.
        timestamp_col="timestamp",            # 시간순 split 검증에 사용할 timestamp 컬럼명이다.
        label_col=LABEL_COL,                  # 정답 label 컬럼명이다.
        split_col="split",                    # 기존 train/val/test 구분 컬럼명이다.
    )
    
    # 5. row_counts에 원본 전체 행수와 train/val/test 행수가 모두 들어있는지 확인한다.
    required_row_count_keys = {"all", "train", "val", "test"}
    missing_row_count_keys = required_row_count_keys - set(result.row_counts)
    
    # 6. 필요한 count key가 없으면 행수 정합성 검증이 불가능하므로 즉시 중단한다.
    if missing_row_count_keys:
        raise ValueError(
            "split export result is missing row count keys. "
            f"missing={sorted(missing_row_count_keys)}, row_counts={result.row_counts}"
        )
    # 7. 원본 단일 parquet의 전체 행수를 가져온다.
    source_row_count = int(result.row_counts["all"])
    
    # 8. export된 train/val/test split 3개의 행수 합계를 계산한다.
    split_row_count = sum(int(result.row_counts[split_name]) for split_name in ["train", "val", "test"])
    
    # 9. split 3개 행수 합계가 원본 전체 행수와 다르면 중단한다.
    if split_row_count != source_row_count:
        raise ValueError(
            "split row count mismatch after export. "
            f"source_rows={source_row_count}, "
            f"train={result.row_counts['train']}, "
            f"val={result.row_counts['val']}, "
            f"test={result.row_counts['test']}, "
            f"split_sum={split_row_count}"
        )
        
    print("row_counts:", result.row_counts)              # 10. split별 row 수를 출력한다.
    print("train_path:", result.output_paths.train_path) # 11. 생성된 train parquet 경로를 출력한다.
    print("val_path  :", result.output_paths.val_path)   # 12. 생성된 validation parquet 경로를 출력한다.
    print("test_path :", result.output_paths.test_path)  # 13. 생성된 test parquet 경로를 출력한다.
    display(hide_test_label_distribution(result.split_summary))                        # 14. split별 기간, label 분포 등 내부 검증 요약을 노트북 표로 확인한다.
    return result                                        # 15. 생성 경로, row count, split summary를 재사용할 수 있도록 결과 객체를 반환한다.
  
# 16. split export를 실행하고 결과 객체를 받아온다. RUN_SPLIT_EXPORT=False이면 None이 된다.
split_export_result = export_split_parquets() 

In [ ]:
def prepare_feature_columns_candidate() -> Path:
    """
    fb_outputs에 저장할 feature column CSV 후보 사본을 준비한다.
    경로 정책
    --------
    - 원본:
      FEATURE_COLUMNS_SOURCE_FILE_PATH
      예: {FEATURE_COLUMNS_SOURCE_DIR}/{FEATURE_COLUMNS_SOURCE_FILE}
    - export 사본:
      FB_OUTPUT_FEATURE_COLUMNS_PATH
      예: {FB_OUTPUT_DIR}/{ARTIFACT_PREFIX}_ml_feature_columns.csv
    사용 목적
    --------
    - Drive의 원본 ml_feature_columns.csv는 전처리 산출물이므로 직접 수정하지 않는다.
    - 이 단계에서는 `fb_outputs`에 검토용 CSV 후보를 만들고, 검토/수정 후 `ml_inputs`에 _approve.csv로 저장해 ML 학습에 사용한다.
    - export 사본의 used_in_ml 표기를 "TRUE" / "FALSE" 문자열 정책으로 정규화한다.
    동작 방식
    --------
    - COPY_FEATURE_COLUMNS=True:
      Drive 원본 CSV를 local export output directory로 복사한다.
    - COPY_FEATURE_COLUMNS=False:
      사용자가 이미 준비한 FB_OUTPUT_FEATURE_COLUMNS_PATH를 그대로 사용한다.
    - 두 경우 모두 마지막에 normalize_feature_columns_file()을 호출해
      used_in_ml 표기를 "TRUE" / "FALSE"로 통일한다.
    overwrite 정책
    ---------------
    - COPY_FEATURE_COLUMNS=True일 때 기존 export CSV가 있으면
      OVERWRITE_POLICY["feature_columns_copy"]에 따라 중단하거나 덮어쓴다.
    - normalize_feature_columns_file()은 export 사본만 수정한다.
      Drive 원본은 수정하지 않는다.
    반환
    ----
    - 후속 preview_input_contract()에서 검증할 fb_outputs 후보 feature column CSV 경로
    - train_val_test에는 사람이 검토/수정 후 ml_inputs에 둔 _approve.csv 경로를 사용한다.
    """
    if COPY_FEATURE_COLUMNS:
        # 1. Drive 원본 feature column CSV가 실제로 존재하는지 먼저 확인한다.
        require_file(FEATURE_COLUMNS_SOURCE_FILE_PATH, "FEATURE_COLUMNS_SOURCE_FILE_PATH")
        
        # 2. local export 사본을 조용히 덮어쓰지 않도록 보호한다.
        guard_output_overwrite(
            FB_OUTPUT_FEATURE_COLUMNS_PATH,
            overwrite=OVERWRITE_POLICY["feature_columns_copy"],
            label="FB_OUTPUT_FEATURE_COLUMNS_PATH",
        )
        
        # 3. export output directory가 없으면 생성한 뒤 Drive 원본 CSV를 local 사본으로 복사한다.
        FB_OUTPUT_FEATURE_COLUMNS_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(FEATURE_COLUMNS_SOURCE_FILE_PATH, FB_OUTPUT_FEATURE_COLUMNS_PATH)
        print("feature_columns copied:", FB_OUTPUT_FEATURE_COLUMNS_PATH)
    else:
        # COPY_FEATURE_COLUMNS=False이면 사용자가 이미 준비한 local export CSV를 사용한다.
        require_file(FB_OUTPUT_FEATURE_COLUMNS_PATH, "FB_OUTPUT_FEATURE_COLUMNS_PATH")
        print(
            "feature_columns copy skipped. Using existing export CSV:",
            FB_OUTPUT_FEATURE_COLUMNS_PATH,
        )
        
    # 4. 기존 CSV에는 True/False bool 표기가 남아 있을 수 있다.
    # 학습 입력 계약은 "TRUE" / "FALSE" 문자열이므로 local export 사본만 정규화한다.
    normalized = normalize_feature_columns_file(
        FB_OUTPUT_FEATURE_COLUMNS_PATH,
        project_root=LOCAL_REPO_ROOT,
        label_col=LABEL_COL,
        overwrite=True,
        strict=True,
    )
    print("feature_columns normalized: used_in_ml uses TRUE/FALSE only")
    print("selected feature_count after normalization:", normalized.selected_count)
    return FB_OUTPUT_FEATURE_COLUMNS_PATH



# fb_outputs 후보 CSV 기준으로 selected feature 목록과 source parquet schema 정합성을 검증한다.
FEATURE_COLUMNS_CANDIDATE_PATH = prepare_feature_columns_candidate()

# 후보 CSV 기준으로 used_in_ml="TRUE" feature 목록과 parquet schema 정합성을 검증한다.
FEATURE_COLUMNS = preview_input_contract(
    input_parquet_path=INPUT_SINGLE_PARQUET_PATH,
    feature_columns_path=FEATURE_COLUMNS_CANDIDATE_PATH,
    label_col=LABEL_COL,
)

# 05_생성 산출물 검증
- 생성된 split parquet 3개 파일이  후속 ML 모듈 입력 계약을 만족하는지 확인한다. 

In [ ]:
def validate_exported_split_file(
    path: Path,
    *,
    expected_split: str,
    feature_columns: list[str],
) -> pd.DataFrame:
    """
    후속 ml_00_ml_*.py 입력으로 사용할 split parquet 하나를 검증한다.
    
    확인 항목
    --------
    - parquet 파일 존재 여부
    - parquet schema에 label, split, selected feature columns가 모두 있는지
    - 파일 내부 split 값이 expected_split 하나로만 구성되어 있는지
    - label 분포와 양 클래스 존재 여부
    
    주의
    ----
    - feature_columns는 ml_feature_columns.csv에서 used_in_ml="TRUE"로 선택된 컬럼 목록이다.
    - 이 함수는 parquet row 전체를 읽기 전에 schema metadata로 컬럼 존재 여부를 먼저 확인한다.
    """
    
    # export된 train/val/test parquet 파일이 실제로 존재하는지 먼저 확인한다.
    require_file(path, f"{expected_split}_parquet")
    
    # parquet 전체를 로드하지 않고 schema metadata에서 컬럼명만 확인한다.
    schema_names = set(parquet_columns(path))
    
    # 후속 ml_00_ml_train.py / ml_00_ml_val.py / ml_00_ml_test.py가 읽기 위해 필요한 최소 컬럼이다.
    # feature_columns, LABEL_COL, split 컬럼이 모두 있어야 한다.
    required_columns = set(feature_columns) | {LABEL_COL, "split"}
    
    # 선택 feature 또는 필수 meta 컬럼이 누락되면 여기서 먼저 차단한다.
    missing = sorted(required_columns - schema_names)
    if missing:
        raise ValueError(
            f"Exported {expected_split} parquet is missing required columns. "
            f"missing={missing[:30]}, missing_count={len(missing)}, path={path}"
        )
        
    # split 값과 label 분포를 확인한다.
    # expected_split="train"이면 파일 내부 split 값도 train만 있어야 한다.
    return summarize_split_labels(path, expected_split=expected_split)


export_summaries = []

# export된 train/val/test 파일을 모두 같은 기준으로 검증한다.
for split_name, split_path in {"train": FB_OUTPUT_TRAIN_PATH, "val": FB_OUTPUT_VAL_PATH, "test": FB_OUTPUT_TEST_PATH}.items():
    summary = validate_exported_split_file(
        split_path,
        expected_split=split_name,
        feature_columns=FEATURE_COLUMNS,
    )
    # 여러 split 요약을 합쳤을 때 어느 파일에서 나온 결과인지 추적하기 위해 path 컬럼을 추가한다.
    summary.insert(0, "path", str(split_path))
    export_summaries.append(summary)



# split별 검증 결과를 하나의 DataFrame으로 합쳐 노트북에서 바로 확인한다.
export_split_summary = pd.concat(export_summaries, ignore_index=True)
display(hide_test_label_distribution(export_split_summary))
print("export validation: OK")

# 06_Manifest 저장
- `fb_outputs` 후보 산출물 경로, `ml_inputs` 복사 대상 경로, feature fingerprint를 JSON으로 저장한다.

In [ ]:
def save_manifest() -> dict:
    """
    후속 ML 실행에 필요한 입력 경로와 검증 요약을 manifest JSON으로 저장한다.
    사용 목적
    --------
    - 이 노트북이 어떤 원본 parquet와 어떤 ml_feature_columns.csv 사본을 사용했는지 기록한다.
    - 후속 ml_00_ml_train.py / ml_00_ml_val.py / ml_00_ml_test.py에 넘길 입력 경로를 한곳에 남긴다.
    - feature 목록과 feature hash를 저장해 학습 시점 feature 순서 재현성을 확인할 수 있게 한다.
    - split별 label 분포 요약을 함께 저장해 train/val/test 입력 검증 결과를 추적한다.
    
    주의
    ----
    - 이 manifest는 학습 결과가 아니다.
    - 이 노트북은 XGBoost 학습, validation threshold 선택, final test를 실행하지 않는다.
    - 기존 manifest가 있으면 overwrite 정책에 따라 저장을 차단한다.
    """
    # 후속 실행과 재현성 확인에 필요한 경로, feature 목록, split 요약을 하나의 dict로 모은다.
    manifest = {

        "notebook": "ml/ml-00_baseline/feature_build/fb_ml00_baseline_test_run.ipynb",         # 어떤 노트북이 이 manifest를 만들었는지 기록한다.
        "purpose": "Export ML-00 input artifacts from 03_ml_feature_process single parquet.",  # manifest의 목적을 명시
        "manifest_type": "ml_input_export",
        
        # 실행 식별 및 재현성 metadata.
        "seed": SEED,
        "source_parquet_file": SOURCE_PARQUET_FILE,
        "export_experiment_id": EXPORT_EXPERIMENT_ID,
        "fb_outputs_dir": str(ML_00_FB_OUTPUTS_DIR),
        "ml_inputs_copy_target_dir": str(ML_00_ML_INPUTS_DIR),
        "run_id": RUN_ID,
        "artifact_prefix": ARTIFACT_PREFIX,
        
        # 입력 원본과 ML feature 선택 기준 파일의 provenance.
        "input_single_parquet": str(INPUT_SINGLE_PARQUET_PATH),
        "feature_columns_source_dir": str(FEATURE_COLUMNS_SOURCE_DIR),
        "feature_columns_source_file": FEATURE_COLUMNS_SOURCE_FILE,
        "feature_columns_source": str(FEATURE_COLUMNS_SOURCE_FILE_PATH),
        "feature_columns_candidate": str(FB_OUTPUT_FEATURE_COLUMNS_PATH),
        
        # feature build가 fb_outputs에 만든 후보 산출물 경로.
        "fb_outputs": {
            "train_path": str(FB_OUTPUT_TRAIN_PATH),
            "val_path": str(FB_OUTPUT_VAL_PATH),
            "test_path": str(FB_OUTPUT_TEST_PATH),
            "feature_columns_path": str(FB_OUTPUT_FEATURE_COLUMNS_PATH),
            "split_summary_path": str(FB_OUTPUT_SPLIT_SUMMARY_PATH),
            "split_existing_summary_path": str(FB_OUTPUT_SPLIT_BUILD_SUMMARY_PATH),
        },
        
        # 사람이 검토 후 fb_outputs에서 ml_inputs로 복사해야 할 후속 ML 입력 대상 경로.
        "ml_input_copy_targets": {
            "train_path": str(ML_INPUT_TRAIN_PATH),
            "val_path": str(ML_INPUT_VAL_PATH),
            "test_path": str(ML_INPUT_TEST_PATH),
            "feature_columns_path": str(ML_INPUT_FEATURE_COLUMNS_PATH),
        },
        
        # 모델 입력 feature 목록과 순서 fingerprint.
        # feature_columns_hash는 feature 이름뿐 아니라 순서까지 반영한다.
        "feature_count": len(FEATURE_COLUMNS),
        "feature_columns_hash": feature_columns_hash(FEATURE_COLUMNS),
        "feature_columns": FEATURE_COLUMNS,
        
        # 원본 단일 parquet와 export된 split parquet의 label 분포 검증 요약.
        "input_split_summary": ML_00_INPUT_SPLIT_LABEL_SUMMARY.to_dict(orient="records"),
        "export_split_summary": export_split_summary.to_dict(orient="records"),
    }
    
    # SAVE_MANIFEST=False이면 파일 저장 없이 manifest dict만 반환한다.
    if SAVE_MANIFEST:
        # 기존 manifest를 조용히 덮어쓰지 않도록 overwrite 정책을 적용한다.
        guard_output_overwrite(
            FB_OUTPUT_MANIFEST_PATH,
            overwrite=OVERWRITE_POLICY["manifest"],
            label="manifest",
        )
        
        # manifest 저장 경로의 parent directory가 없으면 생성한다.
        FB_OUTPUT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
        
        # 사람이 diff/review하기 쉽도록 indent=2 JSON으로 저장한다.
        FB_OUTPUT_MANIFEST_PATH.write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print("manifest saved:", FB_OUTPUT_MANIFEST_PATH)
    return manifest
ML_INPUT_MANIFEST = save_manifest()

## 7. 후속 ML 모듈 입력 경로
- 이 노트북은 `fb_outputs`에 후보 산출물을 만든다.
- 사람이 검토 후 승인한 train/val/test parquet와 `ml_feature_columns.csv`만 `ml_inputs`로 복사한다.
- 후속 `ml_00_ml_train.py`, `ml_00_ml_val.py`, `ml_00_ml_test.py`에는 `ml_inputs` 경로를 넣는다.

In [ ]:
from IPython.display import HTML, display
def artifact_file_status(name: str, path: Path) -> dict:
    path = Path(path)        # path가 str이어도 Path로 통일한다.
    is_file = path.is_file() # 산출물은 파일이어야 하므로 exists()가 아니라 is_file()로 검증한다.
    if not is_file:          # 파일이 아니면 stat() 결과를 산출물 수정 시간으로 쓰지 않는다.
        return {
            "name": name,
            "exists": path.exists(),
            "is_file": False,
            "modified_at": None,
            "path": str(path),
        }
    stat = path.stat()      # 파일이 존재하고 is_file=True이면 stat()으로 수정 시간을 가져온다.
    return {
        "name": name,
        "exists": True,
        "is_file": True,
        "modified_at": pd.Timestamp.fromtimestamp(stat.st_mtime),
        "path": str(path),
    }
    

def display_status_table(df: pd.DataFrame) -> None:
    """긴 path가 잘리지 않도록 HTML table에 줄바꿈 스타일을 적용해 표시한다."""
    html = df.to_html(index=False, escape=True)
    styled_html = html.replace(
        '<table border="1" class="dataframe">',
        (
            '<table border="1" class="dataframe" '
            'style="table-layout: fixed; width: 100%; word-break: break-all; white-space: normal;">'
        ),
    )
    display(HTML(f'<div style="width: 100%; overflow-x: auto;">{styled_html}</div>'))
    
    
fb_artifact_paths = {
    "train_path": FB_OUTPUT_TRAIN_PATH,
    "val_path": FB_OUTPUT_VAL_PATH,
    "test_path": FB_OUTPUT_TEST_PATH,
    "split_summary_path": FB_OUTPUT_SPLIT_SUMMARY_PATH,
    "split_build_summary_path": FB_OUTPUT_SPLIT_BUILD_SUMMARY_PATH,
    "feature_columns_path": FB_OUTPUT_FEATURE_COLUMNS_PATH,
    "manifest_path": FB_OUTPUT_MANIFEST_PATH,
}

ml_input_copy_targets = {
    "train_path": ML_INPUT_TRAIN_PATH,
    "val_path": ML_INPUT_VAL_PATH,
    "test_path": ML_INPUT_TEST_PATH,
    "feature_columns_path": ML_INPUT_FEATURE_COLUMNS_PATH,
}

fb_artifact_status = pd.DataFrame(
    [artifact_file_status(name, path) for name, path in fb_artifact_paths.items()]
)

ml_input_target_status = pd.DataFrame(
    [artifact_file_status(name, path) for name, path in ml_input_copy_targets.items()]
)

print("fb_outputs 생성 산출물 상태")
display_status_table(fb_artifact_status)
print("ml_inputs 복사 대상 상태")
display_status_table(ml_input_target_status)
print("후속 ml_00_ml_train.py 입력 예시: 사람이 fb_outputs 검토 후 ml_inputs로 복사한 뒤 사용")
print("train_path           =", ML_INPUT_TRAIN_PATH)
print("val_path             =", ML_INPUT_VAL_PATH)
print("feature_columns_path =", ML_INPUT_FEATURE_COLUMNS_PATH)
print()
print("후속 ml_00_ml_val.py 입력 예시")
print("val_path             =", ML_INPUT_VAL_PATH)
print()
print("후속 ml_00_ml_test.py 입력 예시")
print("test_path            =", ML_INPUT_TEST_PATH)
print()
print("추적용 fb_outputs 산출물")
print("split_summary_path       =", FB_OUTPUT_SPLIT_SUMMARY_PATH)
print("split_build_summary_path =", FB_OUTPUT_SPLIT_BUILD_SUMMARY_PATH)
print("manifest_path            =", FB_OUTPUT_MANIFEST_PATH)
print()
print("주의: 이 노트북은 학습/validation/test를 실행하지 않았습니다.")
print("주의: fb_outputs 산출물의 exists, is_file, modified_at, path를 사람이 직접 검증하세요.")
print("주의: 승인된 train/val/test parquet는 ml_inputs 대상 경로로 복사하고, feature column CSV는 _approve.csv 파일명으로 저장한 뒤 후속 ML 입력으로 사용하세요.")